# Data understanding and EDA

This notebook validates the normalized table and temporal split, then runs descriptive EDA on the training partition only. It does not use validation/test to select features or train a model.

The source does not declare a timezone or currency. The canonical timestamp stays timezone-naive from `trans_date_trans_time`; `unix_time` is excluded from normalized data after audit found it seven calendar years stale. Run `python -m src.data.pipeline --stage all` first when the Parquet artifacts are absent.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

cwd = Path.cwd()
ROOT = next((candidate for candidate in (cwd, *cwd.parents) if (candidate / 'src' / 'data').is_dir() and (candidate / 'configs' / 'data_schema.yaml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter from the project root or notebooks/ directory.')
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed" / "transactions.parquet"
SPLITS = ROOT / "data" / "splits"
MANIFEST = SPLITS / "split_manifest.parquet"
required = [PROCESSED, MANIFEST, *(SPLITS / f"{name}.parquet" for name in ("train", "validation", "test"))]
missing = [path.relative_to(ROOT).as_posix() for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Run the raw-to-split pipeline before this notebook. Missing: {missing}")

metadata = json.loads((SPLITS / "split_metadata.json").read_text(encoding="utf-8"))
manifest = pd.read_parquet(MANIFEST)
assert manifest["trans_num"].notna().all() and manifest["trans_num"].is_unique
assert set(manifest["split"]) == {"train", "validation", "test"}
parquet_schema = pq.read_schema(PROCESSED)
assert str(parquet_schema.field("timestamp").type) == "timestamp[ns]"
assert parquet_schema.field("timestamp").type.tz is None
assert "unix_time" not in parquet_schema.names
assert not {"first", "last", "street"} & set(parquet_schema.names)

events = pd.read_parquet(PROCESSED, columns=["trans_num", "timestamp"])
joined = events.merge(manifest, on="trans_num", validate="one_to_one")
assert len(joined) == len(events) == len(manifest)
assert joined.groupby("timestamp", observed=True)["split"].nunique().max() == 1
split_summary = pd.read_csv(ROOT / "results" / "eda" / "split_summary.csv")
display(split_summary)
print({"rows": len(events), "split_hash": metadata["split_hash"], "timestamp_dtype": str(events.timestamp.dtype), "timezone": None})

## Training-only EDA

The following cell regenerates aggregate summaries and ten PNG charts using only `train.parquet`. Card identifiers are not emitted in the summary tables. These results describe this simulated Sparkov dataset and do not establish production performance or causal effects.

In [ ]:
from src.data.eda import run_eda

eda_result = run_eda(
    train_path=SPLITS / "train.parquet",
    output_dir=ROOT / "results" / "eda",
    report_path=ROOT / "docs" / "eda_report.md",
    validate_audit=False,
)
chart_paths = sorted((ROOT / "results" / "eda").glob("*.png"))
assert len(chart_paths) == 10
print({**eda_result, "charts": [path.name for path in chart_paths]})

## Interpretation and handoff

Use `trans_num` as the one-to-one join key and preserve `cc_num` only for within-card history grouping. Any history feature must use only the same card's events with timestamps strictly earlier than the anchor; events sharing a timestamp must be processed as one batch. The target `is_fraud` must never enter history features. Observed training history may be used for later validation/test events under the contract.